<center><h1>Niu_Qiyu_HW3</h1></center>

**Name:** Qiyu Niu  
**GitHub Username:** QiyuNiu  
**USC ID:** 6184595941

## 1. Time Series Classification Part 1: Feature Creation/Extraction

The AReM data set has 88 instances. Each instance is stored in one CSV file
and normally has 480 observations for six time series.

### (a) Download Data

I downloaded the data from the
[UCI AReM repository](https://archive.ics.uci.edu/dataset/366/activity%2Brecognition%2Bsystem%2Bbased%2Bon%2Bmultisensor%2Bdata%2Bfusion).
It contains measurements for seven activities. I saved the data under
`../data/AReM` and use that local copy in this notebook.

### Imports

In [9]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 552
DATA_DIR = Path("../data/AReM").resolve()
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"
print("AReM data directory found.")

AReM data directory found.


Most files are comma-separated, but one file uses spaces and two files have an
extra comma at the end. I handle both separators in the loading function.
`sitting/dataset8.csv` is also missing the observation at 13,500 ms, so I fill
that one row by linear interpolation between the two neighboring rows. I do not
change the original CSV files.

In [10]:
RAW_COLUMNS = [
    "time", "avg_rss12", "var_rss12", "avg_rss13",
    "var_rss13", "avg_rss23", "var_rss23"
]
SERIES_COLUMNS = RAW_COLUMNS[1:]
EXPECTED_TIMES = np.arange(0, 120_000, 250)


def load_instance(path):
    '''Read one AReM file and return 480 clean rows on the expected time grid.'''
    rows = []
    with open(path, "r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip() or line.lstrip().startswith("#"):
                continue
            # Replacing commas also handles the two harmless trailing commas.
            fields = line.replace(",", " ").split()
            if len(fields) != 7:
                raise ValueError(
                    f"{path}, line {line_number}: expected 7 values, "
                    f"found {len(fields)}"
                )
            rows.append([float(value) for value in fields])

    frame = pd.DataFrame(rows, columns=RAW_COLUMNS)
    if frame["time"].duplicated().any():
        raise ValueError(f"Duplicate time stamps in {path}")

    frame = frame.set_index("time").reindex(EXPECTED_TIMES)
    missing_rows = int(frame.isna().any(axis=1).sum())
    if missing_rows:
        frame = frame.interpolate(method="linear", limit_direction="both")

    if frame.isna().any().any() or frame.shape != (480, 6):
        raise ValueError(f"Cleaning failed for {path}; shape={frame.shape}")

    frame.index.name = "time"
    return frame.reset_index(), missing_rows

In [11]:
activity_order = [
    "bending1", "bending2", "cycling", "lying",
    "sitting", "standing", "walking"
]

metadata_rows = []
for activity in activity_order:
    activity_dir = DATA_DIR / activity
    paths = sorted(
        activity_dir.glob("dataset*.csv"),
        key=lambda path: int(re.search(r"(\d+)$", path.stem).group(1)),
    )
    for path in paths:
        dataset_id = int(re.search(r"(\d+)$", path.stem).group(1))
        metadata_rows.append(
            {
                "activity": activity,
                "dataset_id": dataset_id,
                "path": path,
            }
        )

metadata = pd.DataFrame(metadata_rows)
assert len(metadata) == 88, f"Expected 88 instances, found {len(metadata)}"

folder_counts = (
    metadata.groupby("activity", sort=False)
    .size()
    .rename("number_of_instances")
    .to_frame()
)
display(folder_counts)
print(f"Total number of instances: {len(metadata)}")

,number_of_instances
activity,
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


Total number of instances: 88


In [12]:
quality_rows = []
for row in metadata.itertuples(index=False):
    cleaned, missing_rows = load_instance(row.path)
    quality_rows.append(
        {
            "file": f"{row.activity}/{row.path.name}",
            "rows_after_cleaning": len(cleaned),
            "interpolated_rows": missing_rows,
        }
    )

quality_report = pd.DataFrame(quality_rows)
display(quality_report.query("interpolated_rows > 0"))
print(
    "All files have 480 rows after cleaning:",
    quality_report["rows_after_cleaning"].eq(480).all(),
)

,file,rows_after_cleaning,interpolated_rows
50,sitting/dataset8.csv,480,1


All files have 480 rows after cleaning: True


### (b) Test and Train Data

I used the split given in the assignment:

- `dataset1.csv` and `dataset2.csv` from `bending1` and `bending2`;
- `dataset1.csv`, `dataset2.csv`, and `dataset3.csv` from each other activity.

I put all remaining files in the training set. The final split has 69 training
instances and 19 test instances.

In [13]:
def choose_split(activity, dataset_id):
    test_limit = 2 if activity in {"bending1", "bending2"} else 3
    return "test" if dataset_id <= test_limit else "train"


metadata["split"] = [
    choose_split(activity, dataset_id)
    for activity, dataset_id in zip(metadata["activity"], metadata["dataset_id"])
]

split_table = pd.crosstab(
    metadata["activity"], metadata["split"], margins=True
).reindex(activity_order + ["All"])
display(split_table)
assert (metadata["split"] == "test").sum() == 19
assert (metadata["split"] == "train").sum() == 69

split,test,train,All
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


### (c) Feature Extraction

#### i. Research

Time-domain features are calculated directly from the original sequence. Common
examples are:

- minimum, maximum, range, mean, median, and quartiles;
- variance, standard deviation, interquartile range, and median
  absolute deviation;
- skewness and kurtosis;
- root mean square, energy, and peak-to-peak amplitude;
- slope, differences, zero-crossing rate, and autocorrelation.

These are common examples, but the exact choice depends on the problem.
[Christ et al. (2018)](https://doi.org/10.1016/j.neucom.2018.03.067)
discuss automatic feature extraction for time series. A study by
[Cabello et al. (2020)](https://doi.org/10.3390/app10165428)
uses features such as mean, standard deviation, IQR, skewness, and kurtosis.

#### ii. Extraction

For each of the six time series, I calculate minimum, maximum, mean, median,
standard deviation, first quartile, and third quartile. Therefore, each instance
has $6 \times 7 = 42$ features. I use the time column only as the index, so it is
not included as one of the six time series.

In [14]:
def extract_features(frame):
    features = {}
    for series_number, column in enumerate(SERIES_COLUMNS, start=1):
        values = frame[column]
        features[f"min{series_number}"] = values.min()
        features[f"max{series_number}"] = values.max()
        features[f"mean{series_number}"] = values.mean()
        features[f"median{series_number}"] = values.median()
        features[f"std{series_number}"] = values.std(ddof=1)
        features[f"1st_quart{series_number}"] = values.quantile(0.25)
        features[f"3rd_quart{series_number}"] = values.quantile(0.75)
    return features


feature_rows = []
for instance_number, row in enumerate(metadata.itertuples(index=False), start=1):
    frame, _ = load_instance(row.path)
    record = {
        "instance": instance_number,
        "activity": row.activity,
        "dataset_id": row.dataset_id,
        "split": row.split,
    }
    record.update(extract_features(frame))
    feature_rows.append(record)

features = pd.DataFrame(feature_rows)
feature_columns = list(features.columns[4:])

assert features.shape == (88, 46)
assert len(feature_columns) == 42
assert not features[feature_columns].isna().any().any()

print("Feature data shape (including four identifier columns):", features.shape)
display(features.head().round(4))

Feature data shape (including four identifier columns): (88, 46)


,instance,activity,dataset_id,split,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,bending1,1,test,37.25,45.00,40.6248,40.50,1.4770,39.25,42.00,0.0,1.30,0.3586,0.43,0.3226,0.00,0.50,4.0,29.50,19.0409,19.250,4.4630,16.00,23.2500,0.0,7.23,0.8325,0.50,0.9657,0.00,1.120,27.25,38.25,34.3113,35.0,2.1884,33.0000,36.00,0.0,1.92,0.5706,0.43,0.5829,0.0,1.30
1,2,bending1,2,test,38.00,45.67,42.8128,42.50,1.4355,42.00,43.67,0.0,1.22,0.3724,0.47,0.2892,0.00,0.50,2.0,29.50,20.0970,21.000,3.8937,19.00,22.2500,0.0,5.76,0.8764,0.50,1.0473,0.00,1.145,27.67,38.50,33.0246,33.0,1.9953,32.0000,34.50,0.0,3.11,0.5711,0.43,0.6010,0.0,1.30
2,3,bending1,3,train,35.00,47.40,43.9545,44.33,1.5588,43.00,45.00,0.0,1.70,0.4263,0.47,0.3387,0.00,0.50,6.5,29.75,22.1224,23.000,3.0309,19.75,24.0000,0.0,4.44,0.4973,0.43,0.5507,0.00,0.830,29.00,38.50,35.5885,36.0,1.9996,35.3625,36.50,0.0,1.79,0.4933,0.43,0.5135,0.0,0.94
3,4,bending1,4,train,33.00,47.75,42.1798,43.50,3.6707,39.15,45.00,0.0,3.00,0.6960,0.50,0.6309,0.00,1.12,8.5,30.00,22.1836,23.000,3.8105,20.50,24.3725,0.0,5.15,0.9899,0.83,0.9537,0.43,1.300,20.00,38.67,33.4939,35.0,3.8494,30.4575,36.33,0.0,2.18,0.6135,0.50,0.5243,0.0,1.00
4,5,bending1,5,train,33.00,45.75,41.6781,41.75,2.2435,41.33,42.75,0.0,2.83,0.5360,0.50,0.4055,0.43,0.71,3.0,28.25,19.0066,19.125,4.0871,16.50,22.0625,0.0,6.42,0.8419,0.50,0.9288,0.43,1.120,23.67,37.50,29.8571,30.0,2.4110,28.4575,31.25,0.0,1.79,0.3833,0.43,0.3892,0.0,0.50


In [15]:
train_features = features.query("split == 'train'").reset_index(drop=True)
test_features = features.query("split == 'test'").reset_index(drop=True)

print("Training feature matrix:", train_features.shape)
print("Test feature matrix:", test_features.shape)

Training feature matrix: (69, 46)
Test feature matrix: (19, 46)


#### iii. Standard Deviation and 90% Bootstrap Confidence Intervals

First, I calculate the sample standard deviation of each feature across the 88
instances using `ddof=1`. Then I create 5,000 bootstrap samples by sampling the
88 instances with replacement. I calculate the standard deviation for every
bootstrap sample and use the 5th and 95th percentiles for the 90% confidence
interval.

I resample complete instances rather than individual time points because each row
of the feature data represents one instance.

In [16]:
def bootstrap_std_table(feature_data, n_bootstrap=5_000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n_instances = len(feature_data)
    rows = []

    for column in feature_data.columns:
        values = feature_data[column].to_numpy(dtype=float)
        sample_indices = rng.integers(
            0, n_instances, size=(n_bootstrap, n_instances)
        )
        bootstrap_samples = values[sample_indices]
        bootstrap_stds = bootstrap_samples.std(axis=1, ddof=1)

        rows.append(
            {
                "feature": column,
                "estimated_std": values.std(ddof=1),
                "90%_CI_lower": np.percentile(bootstrap_stds, 5),
                "90%_CI_upper": np.percentile(bootstrap_stds, 95),
            }
        )

    return pd.DataFrame(rows).set_index("feature")


bootstrap_results = bootstrap_std_table(features[feature_columns])
display(bootstrap_results.round(4))

,estimated_std,90%_CI_lower,90%_CI_upper
feature,,,
min1,9.5700,8.2747,10.7675
max1,4.3944,3.3032,5.3037
mean1,5.3357,4.7195,5.8753
median1,5.4401,4.7950,5.9885
std1,1.7722,1.5727,1.9507
1st_quart1,6.1536,5.5832,6.6273
3rd_quart1,5.1389,4.3478,5.8314
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6270,5.4064


A wider interval means that the standard deviation estimate has more uncertainty.
These intervals are for the variation of each extracted feature across the 88
instances, not for the 480 raw values inside one file.

#### iv. Select Three Features

I select **minimum, mean, and maximum**. The minimum and maximum show the range
of the signal, while the mean shows its average level. They are simple to
interpret and may help distinguish activities with different signal levels. Other
choices could also be reasonable, but these three are suggested in the assignment.

## 2. ISLR 3.7.4

A linear model and a cubic model are fitted to the same $n=100$ observations:

$$
\text{linear: }Y=\beta_0+\beta_1X+\epsilon,
\qquad
\text{cubic: }Y=\beta_0+\beta_1X+\beta_2X^2+\beta_3X^3+\epsilon.
$$

The linear model is nested inside the cubic model: setting
$\beta_2=\beta_3=0$ makes the cubic model linear.

### (a) True Relationship Is Linear: Training RSS

I expect the cubic regression to have training RSS less than or equal to the
linear regression. The cubic model can produce the same fit as the linear model
by setting the coefficients of $X^2$ and $X^3$ to zero. It can also use those two
terms to fit some of the training noise, so its training RSS will usually be lower.

### (b) True Relationship Is Linear: Test RSS

I expect the linear regression to have lower test RSS on average because the true
relationship is linear. The extra terms in the cubic model are not needed and may
fit noise from the training data. This can increase its variance and hurt its test
performance. A particular test set could give a different result, but the linear
model should be better on average.

### (c) True Relationship Is Nonlinear: Training RSS

I again expect the cubic regression to have training RSS less than or equal to
the linear regression. The cubic model contains the linear model as a special
case, so adding the two polynomial terms cannot increase the least-squares
training RSS.

### (d) True Relationship Is Nonlinear: Test RSS

There is not enough information to tell. If the true relationship is strongly
nonlinear and close to a cubic curve, the cubic model may have lower test RSS. If
the nonlinearity is weak, the simpler linear model may perform better because it
has lower variance. The result depends on the form and strength of the nonlinear
relationship.